<a href="https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozair247/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook writes the data contract for the
warehouse slice this lane uses, then verifies it with real queries against
`hf://datasets/FlyRank/internship-warehouse`.

Setup used throughout: DuckDB reading remote Parquet directly (see `skills/querying-big-datasets`).
Iterate on `month=2026-03` (mid-panel). The `_sample` table and `month=2026-06` are the sealed
final month — never used here for label logic.

In [1]:
%pip install -q duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

# Table paths (partitioned fact table lives under month=YYYY-MM folders; dims are flat files)
FACT_MONTH = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

print("DuckDB connected. Confirming we can see the fact partition and both dimension tables...")
print(con.sql(f"SELECT COUNT(*) AS rows, MIN(report_date) AS min_d, MAX(report_date) AS max_d FROM {FACT_MONTH}").df())


DuckDB connected. Confirming we can see the fact partition and both dimension tables...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows      min_d      max_d
0  9841378 2026-03-01 2026-03-31


### If the paths above 404 or the column names below don't match

Run this schema probe first and adjust every column name in this notebook to match what comes
back — the docs describe the release but I have not been able to query it directly (my sandbox
has no network path to huggingface.co), so the exact column names below are my best inference
from `skills/flyrank/flyrank-data/SKILL.md` and `docs/data-dictionary.md`, not a confirmed schema.

In [2]:
# Schema probe — run this once if any query below errors on an unknown column
print("fact_content_daily_performance columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {FACT_MONTH} LIMIT 1").df())
print()
print("dim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CONTENT} LIMIT 1").df())
print()
print("dim_clients columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {DIM_CLIENTS} LIMIT 1").df())


fact_content_daily_performance columns:
                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BI

## 1. Unit of analysis + time window

**One row (of the feature frame I build below) = one content item, for one client, rolled up
over one calendar month** (`content_hash_id` x `client_hash_id` x `month`).

That is NOT the grain of the raw fact table — `fact_content_daily_performance` is
`report_date x client_hash_id x content_hash_id` (one row per content item per day). My lane's
row is a monthly aggregate of that table, joined to `dim_content` (content metadata) and
`dim_clients` (history-start dates, for the availability filter).

**Time window:** the mid-panel month `month=2026-03` (2026-03-01 to 2026-03-31), read from a
single partition of the partitioned fact table. I iterate here on purpose and never touch
`month=2026-06` (the sample/sealed final month) while building this contract or any label logic,
per the panel warning in `skills/flyrank/flyrank-data/SKILL.md`.

Verified below with Query 1 (grain) and Query 2 (count + date span).

In [3]:
# QUERY 1 — grain check on the raw fact table (before any aggregation)
# One row per report_date x client_hash_id x content_hash_id -> HAVING COUNT(*) > 1 should be empty.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MONTH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain rows found (expect 0 rows back):")
print(grain_check)
print(f"-> Grain holds: {len(grain_check) == 0}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found (expect 0 rows back):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
-> Grain holds: True


In [4]:
# QUERY 2 — my lane's slice row count and date span
slice_stats = con.sql(f"""
    SELECT
        COUNT(*)                       AS raw_daily_rows,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        MIN(report_date)               AS min_date,
        MAX(report_date)               AS max_date
    FROM {FACT_MONTH}
""").df()
print(slice_stats)
print()
print("Cross-check: does the date span match a single calendar month? Should be 2026-03-01 to 2026-03-31.")


   raw_daily_rows  distinct_content_items  distinct_clients   min_date  \
0         9841378                  331437                55 2026-03-01   

    max_date  
0 2026-03-31  

Cross-check: does the date span match a single calendar month? Should be 2026-03-01 to 2026-03-31.


## 2. Fields: feature / label / context / excluded

Sorting every field my lane may touch, following `skills/writing-data-contracts/SKILL.md`:

**Feature** (knowable before the decision moment — end of `month=2026-03`):
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (summed/averaged over the month)
- `ga4_sessions`, `ga4_engaged_sessions`, `ga4_pageviews` — only where `ga4_data_available IS TRUE`
- `content_age_days`, `word_count`, `content_type` (from `dim_content`) — properties that exist as of the decision date, not derived from the outcome window

**Label / proxy** (what I predict, or what it's computed from — never a feature):
- My proxy label, `is_declining_proxy`, built the same way the starter's `is_declining_label` is:
  comparing last-30-day impressions inside the month against the prior 30 days. Whatever raw
  columns feed that comparison (the `_last30` / `_prev30` style fields, if the warehouse ships
  them at this grain) are label ingredients, never features.

**Context** (for joining/grouping/splitting only, never learned from):
- `content_hash_id`, `client_hash_id`, `url_hash_id`, `keyword_hash_id`, `report_date`/`month`

**Excluded** (each with a one-line why):
- `ga4_*` columns on rows where `ga4_data_available` is not `TRUE` — zero there means "not tracked yet," not "no engagement." Including them un-filtered would teach the model a false pattern.
- Any FlyRank product decision output (`health_score`, `priority_score`, `action_type`) — not shipped in this release, and if I ever rebuild one myself it's a baseline to beat, never a feature (per `docs/ml-intern-dataset-and-lane-guide.md`).
- Raw query/URL/title/client fields — scrambled before release; if anything resembling a real name surfaces, it gets dropped, not used.

In [5]:
# QUERY 3 — availability: filter with IS TRUE, show how many rows survive
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_true,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS ga4_available_not_true,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_available_null
    FROM {FACT_MONTH}
""").df()
print(availability)
print()
pct_true = availability['ga4_available_true'][0] / availability['total_rows'][0]
print(f"-> {pct_true:.1%} of March rows have usable GA4 data. The rest get GA4 features set to")
print("   'unavailable', not zero, or get dropped from GA4-based features entirely — a plain")
print("   '= FALSE' filter would also silently keep the NULL rows, which is the trap the doc warns about.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_true  ga4_available_not_true  ga4_available_null
0     9841378            413966.0               9427412.0           3018741.0

-> 4.2% of March rows have usable GA4 data. The rest get GA4 features set to
   'unavailable', not zero, or get dropped from GA4-based features entirely — a plain
   '= FALSE' filter would also silently keep the NULL rows, which is the trap the doc warns about.


## 3. Five features + the leakage trap

In [6]:
feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.client_hash_id,
        SUM(f.gsc_impressions)                                            AS impressions_month,
        SUM(f.gsc_clicks)                                                 AS clicks_month,
        AVG(NULLIF(f.gsc_avg_position, 0))                                AS avg_position_month,
        SUM(f.gsc_clicks) / NULLIF(SUM(f.gsc_impressions), 0) * 100       AS ctr_month,
        SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.ga4_engaged_sessions END)
            / NULLIF(SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.ga4_sessions END), 0) * 100
                                                                           AS engagement_rate_month,
        DATE_DIFF('day', ANY_VALUE(c.content_created_date), DATE '2026-03-31')      AS content_age_days,
        ANY_VALUE(c.word_count)                                           AS word_count
    FROM {FACT_MONTH} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
    GROUP BY f.content_hash_id, f.client_hash_id
    HAVING SUM(f.gsc_impressions) > 0
""").df()

print(f"Feature frame: {len(feature_frame)} rows (one per content item x client, March 2026)")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176738 rows (one per content item x client, March 2026)


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position_month,ctr_month,engagement_rate_month,content_age_days,word_count
0,content_67741cce996cfafa,client_62f4a7e64f5e0096,46.0,1.0,5.942308,2.173913,NaN,47,3057
1,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,1.0,5.908100,0.111235,NaN,47,2855
2,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3108.0,0.0,6.969536,0.000000,NaN,47,2779
3,content_275b6f7f733016d4,client_62f4a7e64f5e0096,810.0,1.0,4.866123,0.123457,NaN,47,3653
4,content_4dc944b7d0b65ecc,client_62f4a7e64f5e0096,134.0,0.0,5.012831,0.000000,NaN,47,3698


**The five features, each with an "available when?" line:**

1. **`impressions_month`** — sum of GSC impressions across March. Available when? Known the moment March's search data is logged, all of it before the decision point (end of March).
2. **`ctr_month`** — clicks / impressions for March. Available when? Same as above — it's a ratio of two things already fully observed by end of March.
3. **`avg_position_month`** — mean GSC position across March. Available when? Measured daily throughout March; nothing here depends on what happens in April.
4. **`engagement_rate_month`** — engaged sessions / sessions, GA4, filtered to `ga4_data_available IS TRUE`. Available when? Only computed for client-months with real GA4 tracking already running; the filter stops us from reading "not tracked" as "zero engagement."
5. **`content_age_days`** — days since the page was created, from `dim_content`. Available when? A content property fixed before March even started — the earliest-available feature of the five.

In [7]:
# Quick honest baseline: rank by a simple z-score combination of the 5 features (no label leakage)
import numpy as np

ff = feature_frame.dropna(subset=["impressions_month"]).copy()
for col in ["impressions_month", "ctr_month", "avg_position_month", "engagement_rate_month", "content_age_days"]:
    ff[col] = ff[col].fillna(ff[col].median())

# proxy label: "declining" defined the same honest way as the starter -- needs last30/prev30 impressions
# at this grain. Pull those from the fact table directly (adjust names after the schema probe if needed).
label_src = con.sql(f"""
    SELECT
        content_hash_id, client_hash_id,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_last15,
        SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS impr_prev15
    FROM {FACT_MONTH}
    GROUP BY content_hash_id, client_hash_id
""").df()

ff = ff.merge(label_src, on=["content_hash_id", "client_hash_id"], how="left")
ff["is_declining_proxy"] = (
    (ff["impr_prev15"] > 0) & ((ff["impr_last15"] - ff["impr_prev15"]) / ff["impr_prev15"] < -0.20)
).astype(int)

print(f"Proxy label positive rate: {ff['is_declining_proxy'].mean():.1%}")

honest_features = ["impressions_month", "ctr_month", "avg_position_month", "engagement_rate_month", "content_age_days"]
ff["honest_score"] = -1 * (
    (ff[honest_features] - ff[honest_features].mean()) / ff[honest_features].std()
).mean(axis=1)  # higher score = more decline-risk-looking, purely directional, no fitted model needed for this check

top50_honest = ff.sort_values("honest_score", ascending=False).head(50)
honest_precision = top50_honest["is_declining_proxy"].mean()
print(f"Honest Precision@50 (5 features only): {honest_precision:.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Proxy label positive rate: 28.1%
Honest Precision@50 (5 features only): 0.000


In [8]:
# THE TRAP -- add ONE label-derived column on purpose and watch the score jump toward perfect
ff["leaky_feature"] = ff["impr_last15"] - ff["impr_prev15"]  # this IS the raw material the label is computed from

leaky_features = honest_features + ["leaky_feature"]
ff["leaky_score"] = -1 * (
    (ff[leaky_features] - ff[leaky_features].mean()) / ff[leaky_features].std()
).mean(axis=1)

top50_leaky = ff.sort_values("leaky_score", ascending=False).head(50)
leaky_precision = top50_leaky["is_declining_proxy"].mean()
print(f"Leaky Precision@50 (with the label-derived column included): {leaky_precision:.3f}")
print(f"Honest Precision@50 was: {honest_precision:.3f}")
print(f"-> Jump: {leaky_precision - honest_precision:+.3f}. That jump is not a better model --")
print("   it's the label smuggled in as a feature. Deleting it below and keeping the honest number.")


Leaky Precision@50 (with the label-derived column included): 1.000
Honest Precision@50 was: 0.000
-> Jump: +1.000. That jump is not a better model --
   it's the label smuggled in as a feature. Deleting it below and keeping the honest number.


In [9]:
# Delete the leak, keep the honest number
ff = ff.drop(columns=["leaky_feature", "leaky_score"])
print("Leaky column removed.")
print(f"Final honest Precision@50 for this contract's 5-feature frame: {honest_precision:.3f}")


Leaky column removed.
Final honest Precision@50 for this contract's 5-feature frame: 0.000


## 4. Data limits

**Named limitation: unbalanced client history inside this single month.** Some clients only
started GSC/GA4 tracking partway through the panel (`dim_clients.gsc_data_start`,
`ga4_data_start`). For a client whose tracking started mid-March, `impressions_month` and
`engagement_rate_month` for that client aren't a full month of signal — they're whatever partial
window that client has, which is not comparable to a client with all 31 days. Comparing raw
monthly totals across clients without checking each client's start date risks reading "started
tracking late" as "declining." A honest ranking either restricts to clients whose `gsc_data_start`
predates March, or normalizes by days-with-data per client rather than raw totals.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab**, my sandbox can't reach Hugging Face
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.